# Featuresmith Tutorial: Target Leakage & DS Workflows

In this tutorial, we will use the **Customer Churn** dataset to demonstrate how Featuresmith serves as the reasoning layer to detect target leakage before training estimators.

---

In [ ]:
import os

import featuresmith as fs

dataset_path = os.path.join("..", "data", "processed", "customer_churn.csv")
dataset = fs.load(dataset_path)

### Step 1: Detect Potential Target Leakage

Target leakage happens when a column carries information about the target that is unavailable during real inference. Let's analyze the Customer Churn dataset, specifying the target label `churn_label`.

In [ ]:
result = fs.analyze(dataset, target_column="churn_label")

print("Evaluating Rule Findings:")
leakage_findings = [
    f for f in result.findings if f.rule_id == "leakage.potential_leakage"
]

print(f"Detected leakage findings: {len(leakage_findings)}")
for finding in leakage_findings:
    print(f"- Column: {finding.column_name} | Severity: {finding.severity.upper()}")
    print(f"  Title: {finding.title}")
    print(f"  Reason: {finding.description}")

### Step 2: Inspecting Evidence

Let's look at the correlation coefficients that triggered this leakage finding.

In [ ]:
for finding in leakage_findings:
    print(f"Evidence details for column '{finding.column_name}':")
    # Finding.evidence dictionary contains the exact Pearson score calculated
    print(
        f"- Evidence details: {finding.evidence if hasattr(finding, 'evidence') else 'Pearson >= 0.99'}"
    )

### Step 3: Best Practice Data Science Pre-modeling Workflow

Before passing your dataframe to a scikit-learn model, run an automated Featuresmith audit function to drop leaking or high cardinality columns:

In [ ]:
import pandas as pd


def prepare_modeling_data(file_path: str, target: str) -> pd.DataFrame:
    # 1. Run featuresmith audit
    dataset = fs.load(file_path)
    result = fs.analyze(dataset, target_column=target)

    # 2. Extract columns flagged as critical (e.g. potential leakage or empty)
    critical_cols = set()
    for finding in result.findings:
        if finding.severity.lower() == "critical":
            critical_cols.add(finding.column_name)

    print(f"Dropping critical/leaky columns: {critical_cols}")

    # 3. Load pandas df and drop columns
    df = pd.read_csv(file_path)
    df_cleaned = df.drop(columns=list(critical_cols))

    return df_cleaned


df_clean = prepare_modeling_data(dataset_path, "churn_label")
print(f"\nPrepared dimensions: {df_clean.shape}")